# V2+WC vs V3+WC — Best Configs Comparison

**Strategy A:** IchiV2+WC — oi=5%, pool=4%, static-107, 10 slots (Exp 014 best)  
**Strategy B:** IchiV3+WC — oi=5%, pool=4%, static, 10 slots (Exp 004 best)  
**Strategy C:** V3+WC — oi=2.5%, pool=2%, static-107, 10 slots (currently deployed)  
**Benchmark:** BTC buy-and-hold  
**Starting balance:** $100,000  
**Timerange:** 2021-07-18 to 2026-03-12

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

STRATEGIES = {
    'V2+WC oi=5% (Exp 014)': 'experiments/ichiv2-gmx/014-dynamic-sizing-sweep/results/pool0.04_oi0.05.zip',
    'V3+WC oi=5% (Exp 004)': 'experiments/ichiv3-gmx/004-dynamic-sizing-sweep/results/pool0.04_oi0.05_ratio0.05.zip',
    'V3+WC oi=2.5% [LIVE]':  'experiments/ichiv2-gmx/020-v2wc-vs-v3wc-final/results/arm_d_v3wc_best.zip',
}

STARTING_BALANCE = 100_000
STAKE_CURRENCY = 'USDC'

COLORS = {
    'V2+WC oi=5% (Exp 014)': '#2980b9',
    'V3+WC oi=5% (Exp 004)': '#e67e22',
    'V3+WC oi=2.5% [LIVE]':  '#95a5a6',
}

HTML_OUTPUT = 'v2_vs_v3_best_comparison_report.html'

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
from pathlib import Path
import zipfile
import json
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import quantstats as qs
qs.extend_pandas()

REPO_ROOT = Path('/home/ubuntu/dev/gmx-ccxt-freqtrade')
os.chdir(REPO_ROOT)
None

In [ ]:
# ============================================================
# LOAD TRADE DATA FROM BACKTEST ZIPS
# ============================================================

def load_backtest(zp_path):
    """Extract trades + strategy summary from a backtest zip."""
    with zipfile.ZipFile(zp_path) as z:
        jf = [n for n in z.namelist() if n.endswith('.json') and 'meta' not in n][0]
        data = json.load(z.open(jf))
    strat_key = list(data['strategy'].keys())[0]
    summary = data['strategy'][strat_key]
    trades = pd.DataFrame(summary['trades'])
    # Filter out force_exit boundary trades
    trades = trades[trades['exit_reason'] != 'force_exit'].copy()
    trades['close_date'] = pd.to_datetime(trades['close_date'])
    trades['open_date'] = pd.to_datetime(trades['open_date'])
    return trades, summary

def build_equity(trades, starting_balance):
    """Build daily equity curve from trade list."""
    ts = trades.sort_values('close_date').copy()
    ts['equity'] = starting_balance + ts['profit_abs'].cumsum()
    ts = ts.set_index('close_date')
    daily = ts['equity'].resample('D').last().ffill()
    dr = pd.date_range(start=daily.index.min(), end=daily.index.max(), freq='D')
    daily = daily.reindex(dr).ffill()
    if daily.iloc[0] != starting_balance:
        first = daily.index[0] - pd.Timedelta(days=1)
        daily = pd.concat([pd.Series([starting_balance], index=[first]), daily])
    return daily

def build_returns(equity):
    """Convert equity series to daily returns."""
    r = equity.pct_change().dropna()
    r.index = r.index.tz_localize(None)
    return r

# Load all strategies
all_data = {}
for name, zp in STRATEGIES.items():
    trades, summary = load_backtest(zp)
    equity = build_equity(trades, STARTING_BALANCE)
    returns = build_returns(equity)
    all_data[name] = {'trades': trades, 'summary': summary, 'equity': equity, 'returns': returns}

# Load BTC benchmark
first_returns = list(all_data.values())[0]['returns']
benchmark = qs.utils.download_returns('BTC-USD', period='max')
benchmark = benchmark.loc[first_returns.index.min():first_returns.index.max()]
benchmark.index = benchmark.index.tz_localize(None)
benchmark.name = 'BTC'

btc_equity = STARTING_BALANCE * (1 + benchmark).cumprod()

print(f'Loaded {len(all_data)} strategies + BTC benchmark')
for name, d in all_data.items():
    n = len(d['trades'])
    final = d['equity'].iloc[-1]
    print(f'  {name}: {n} trades, final equity ${final:,.0f}')

---
## Summary Metrics Comparison

In [ ]:
from IPython.display import display, HTML

rows = []
for name, d in all_data.items():
    r = d['returns']
    t = d['trades']
    s = d['summary']
    eq = d['equity']
    
    wins = len(t[t['profit_abs'] > 0])
    losses = len(t[t['profit_abs'] <= 0])
    win_pnl = t[t['profit_abs'] > 0]['profit_abs'].sum()
    loss_pnl = t[t['profit_abs'] <= 0]['profit_abs'].sum()
    pf = win_pnl / abs(loss_pnl) if loss_pnl != 0 else 0
    
    rows.append({
        'Strategy': name,
        'Profit %': f"{s['profit_total']*100:.1f}%",
        'CAGR %': f"{s.get('cagr',0)*100:.1f}%",
        'Sharpe': f"{qs.stats.sharpe(r):.2f}",
        'Sortino': f"{qs.stats.sortino(r):.2f}",
        'Calmar': f"{s.get('calmar',0):.2f}",
        'Max DD %': f"{s['max_drawdown_account']*100:.1f}%",
        'Volatility %': f"{qs.stats.volatility(r)*100:.1f}%",
        'Trades': len(t),
        'Wins': wins,
        'Losses': losses,
        'Win %': f"{wins/len(t)*100:.1f}%",
        'Profit Factor': f"{pf:.2f}",
        'Gross Win $': f"${win_pnl:,.0f}",
        'Gross Loss $': f"${loss_pnl:,.0f}",
        'Avg Win $': f"${win_pnl/wins:,.0f}" if wins else '$0',
        'Avg Loss $': f"${loss_pnl/losses:,.0f}" if losses else '$0',
        'Final Equity $': f"${eq.iloc[-1]:,.0f}",
    })

df_summary = pd.DataFrame(rows).set_index('Strategy').T
display(df_summary.style.set_caption('Strategy Comparison — Key Metrics').set_table_styles([
    {'selector': 'caption', 'props': [('font-size', '18px'), ('font-weight', 'bold'), ('padding', '10px')]},
    {'selector': 'th', 'props': [('background-color', '#f0f0f0'), ('padding', '8px'), ('font-size', '13px')]},
    {'selector': 'td', 'props': [('padding', '6px 12px'), ('font-size', '13px')]},
]))

---
## Equity Curves

In [ ]:
fig, ax = plt.subplots(figsize=(18, 9))

for name, d in all_data.items():
    eq = d['equity']
    ax.plot(eq.index, eq.values, linewidth=2.2, label=name, color=COLORS[name])

# BTC benchmark
ax.plot(btc_equity.index, btc_equity.values, linewidth=1.5, label='BTC Buy & Hold',
        color='#bdc3c7', linestyle='--', alpha=0.7)

ax.axhline(y=STARTING_BALANCE, color='gray', linestyle=':', linewidth=0.8, alpha=0.5)
ax.set_title('Equity Curves — V2+WC vs V3+WC Best Configs', fontsize=20, fontweight='bold', pad=20)
ax.set_ylabel(f'Portfolio Value ({STAKE_CURRENCY})', fontsize=13)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'${x:,.0f}'))
ax.legend(fontsize=13, loc='upper left')
ax.grid(True, alpha=0.2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

---
## Drawdown Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(18, 6))

for name, d in all_data.items():
    eq = d['equity']
    running_max = eq.cummax()
    dd = (eq - running_max) / running_max * 100
    ax.fill_between(dd.index, dd.values, 0, alpha=0.3, color=COLORS[name])
    ax.plot(dd.index, dd.values, linewidth=1.5, label=name, color=COLORS[name])

ax.set_title('Drawdown Comparison', fontsize=20, fontweight='bold', pad=20)
ax.set_ylabel('Drawdown (%)', fontsize=13)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x:.0f}%'))
ax.legend(fontsize=13, loc='lower left')
ax.grid(True, alpha=0.2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

---
## Monthly Returns Heatmaps

In [ ]:
for name, d in all_data.items():
    r = d['returns']
    monthly = r.resample('M').apply(lambda x: (1+x).prod() - 1) * 100
    df_m = pd.DataFrame({'ret': monthly})
    df_m['year'] = df_m.index.year
    df_m['month'] = df_m.index.month
    pivot = df_m.pivot_table(index='year', columns='month', values='ret', aggfunc='sum')
    pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'][:len(pivot.columns)]
    
    fig, ax = plt.subplots(figsize=(16, max(3, len(pivot)*0.6+1)))
    vmax = np.percentile(np.abs(pivot.values[~np.isnan(pivot.values)]), 95)
    sns.heatmap(pivot, cmap='RdYlGn', center=0, annot=True, fmt='.1f',
                linewidths=0.5, linecolor='white', vmin=-vmax, vmax=vmax, ax=ax,
                cbar_kws={'label': 'Monthly Return (%)', 'shrink': 0.8},
                annot_kws={'size': 9})
    ax.set_title(f'{name} — Monthly Returns (%)', fontsize=16, fontweight='bold', pad=12)
    ax.set_ylabel('')
    plt.tight_layout()
    plt.show()

---
## Rolling Sharpe (252-day)

In [ ]:
fig, ax = plt.subplots(figsize=(18, 6))

for name, d in all_data.items():
    r = d['returns']
    rolling_sharpe = r.rolling(252).apply(lambda x: x.mean() / x.std() * np.sqrt(365) if x.std() > 0 else 0)
    ax.plot(rolling_sharpe.index, rolling_sharpe.values, linewidth=1.8, label=name, color=COLORS[name])

ax.axhline(y=0, color='gray', linestyle=':', linewidth=0.8)
ax.axhline(y=1, color='green', linestyle=':', linewidth=0.5, alpha=0.5)
ax.axhline(y=2, color='green', linestyle=':', linewidth=0.5, alpha=0.3)
ax.set_title('Rolling 252-Day Sharpe Ratio', fontsize=20, fontweight='bold', pad=20)
ax.set_ylabel('Sharpe Ratio', fontsize=13)
ax.legend(fontsize=13)
ax.grid(True, alpha=0.2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

---
## Position Size Distribution

In [ ]:
fig, axes = plt.subplots(1, len(all_data), figsize=(6*len(all_data), 5), sharey=True)
if len(all_data) == 1:
    axes = [axes]

for ax, (name, d) in zip(axes, all_data.items()):
    stakes = d['trades']['stake_amount']
    ax.hist(stakes, bins=50, color=COLORS[name], alpha=0.7, edgecolor='white')
    ax.axvline(stakes.median(), color='red', linestyle='--', linewidth=1.5, label=f'Median ${stakes.median():,.0f}')
    ax.set_title(name, fontsize=14, fontweight='bold')
    ax.set_xlabel('Stake Amount ($)', fontsize=11)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'${x:,.0f}'))
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.2)

axes[0].set_ylabel('Number of Trades', fontsize=11)
fig.suptitle('Position Size Distribution', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Per-Pair PnL Comparison (Top 15 pairs)

In [ ]:
# Get union of top pairs across all strategies
all_pair_pnl = {}
for name, d in all_data.items():
    pair_pnl = d['trades'].groupby('pair')['profit_abs'].sum().sort_values(ascending=False)
    all_pair_pnl[name] = pair_pnl

# Top 15 by average PnL across strategies
combined = pd.DataFrame(all_pair_pnl).fillna(0)
combined['avg'] = combined.mean(axis=1)
top_pairs = combined.sort_values('avg', ascending=False).head(15).index.tolist()

fig, ax = plt.subplots(figsize=(18, 8))
x = np.arange(len(top_pairs))
width = 0.25

for i, (name, pnl) in enumerate(all_pair_pnl.items()):
    vals = [pnl.get(p, 0) for p in top_pairs]
    ax.bar(x + i*width, vals, width, label=name, color=COLORS[name], alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels([p.replace('/USDC:USDC','') for p in top_pairs], rotation=45, ha='right', fontsize=11)
ax.set_ylabel('Total PnL ($)', fontsize=13)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'${x:,.0f}'))
ax.axhline(y=0, color='gray', linewidth=0.5)
ax.set_title('Per-Pair PnL — Top 15 Pairs', fontsize=18, fontweight='bold', pad=15)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.2, axis='y')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

---
## QuantStats Tearsheets

Individual tearsheets for each strategy.

In [ ]:
for name, d in all_data.items():
    print(f'\n{"="*60}')
    print(f'  {name}')
    print(f'{"="*60}')
    qs.reports.full(d['returns'], benchmark=benchmark, benchmark_title='BTC', rf=0.0, title=name)

---
## QuantStats Metrics Side-by-Side

In [ ]:
all_metrics = {}
for name, d in all_data.items():
    m = qs.reports.metrics(d['returns'], benchmark=benchmark, rf=0.0, display=False)
    # Keep only Strategy column
    all_metrics[name] = m['Strategy'] if 'Strategy' in m.columns else m.iloc[:, 0]

comparison = pd.DataFrame(all_metrics)
display(comparison.style.set_caption('QuantStats Metrics — Side by Side').set_table_styles([
    {'selector': 'caption', 'props': [('font-size', '18px'), ('font-weight', 'bold'), ('padding', '10px')]},
    {'selector': 'th', 'props': [('background-color', '#f0f0f0'), ('padding', '8px')]},
    {'selector': 'td', 'props': [('padding', '4px 8px')]},
]))

---
*Comparison generated from backtest data. Force-exited trades excluded. Benchmark: BTC buy-and-hold.*  
*Backtests are not indicative of future performance.*